# Chicago Taxi Data Download Pipeline -> Volume

This notebook downloads monthly taxi trip data from the City of Chicago's open data portal. It places them into a volume.

In [0]:
import requests
import pandas as pd
import os
from typing import Optional
from io import StringIO
from concurrent.futures import ThreadPoolExecutor, as_completed, wait, FIRST_COMPLETED

BASE_URL = "https://data.cityofchicago.org/resource/wrvz-psew.csv"
OUTPUT_DIR = "/Volumes/chicago_city/raw/trips_raw"
LIMIT = 1000
MAX_WORKERS = 40

MONTHS = [
    ("2023-01", "2023-01-01T00:00:00", "2023-02-01T00:00:00"),
    ("2023-02", "2023-02-01T00:00:00", "2023-03-01T00:00:00"),
    ("2023-03", "2023-03-01T00:00:00", "2023-04-01T00:00:00"),
    ("2023-04", "2023-04-01T00:00:00", "2023-05-01T00:00:00"),
]

def fetch_page(start_date: str, end_date: str, offset: int) -> pd.DataFrame:
    url = (
        f"{BASE_URL}"
        f"?$where=trip_start_timestamp between '{start_date}' and '{end_date}'"
        f"&$limit={LIMIT}&$offset={offset}&$order=trip_start_timestamp")

    try: 
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        df = pd.read_csv(StringIO(r.text))
        return df
    except Exception as e:
        print(e)
        return None
    return df

def download_month(year_month: str, start_date: str, end_date: str) -> tuple[str, Optional[pd.DataFrame]]:
    fp = fetch_page(start_date, end_date, offset=0)
    if fp is None or fp.empty:
        print("No data on first page, let's skip it.")
        return year_month, None
    pages: dict[int, pd.DataFrame] = {0: fp}

    if len(fp) == LIMIT:
        inwo = min(10, MAX_WORKERS)
        offset = LIMIT

        with ThreadPoolExecutor(max_workers= inwo) as inner:
            pending: dict = {}
            BATCH = inwo * 2
            for _ in range(BATCH):
                fut = inner.submit(fetch_page, start_date, end_date, offset)
                pending[fut] = offset
                offset += LIMIT

            while pending:
                done, _ = wait(pending, return_when=FIRST_COMPLETED)
                for fut in done:
                    pg_offset = pending.pop(fut)
                    df_page = fut.result()
                    if df_page is None:
                        continue
                    pages[pg_offset] = df_page
                    if len(df_page) == LIMIT:
                        fut2 = inner.submit(fetch_page, start_date, end_date, offset)
                        pending[fut2] = offset
                        offset += LIMIT
    
    order_dfs = [pages[k] for k in sorted(pages)]
    findf = pd.concat(order_dfs, ignore_index= True)
    return year_month, findf

def save_parquet(year_month: str, df: pd.DataFrame) -> str:
    os.makedirs(OUTPUT_DIR,exist_ok=True)
    path = f"{OUTPUT_DIR}/trips_{year_month}.parquet"
    df.to_parquet(path, index=False, engine="pyarrow", compression="snappy")
    return path

# Top-level pool: one thread per month
with ThreadPoolExecutor(max_workers=len(MONTHS)) as executor:
    futures = {
        executor.submit(download_month, ym, s, e): ym
        for ym, s, e in MONTHS
    }
    results: dict[str, pd.DataFrame] = {}
    for fut in as_completed(futures):
        ym, df = fut.result()
        if df is not None:
            results[ym] = df

# Save in sorted order for reproducibility
saved_paths = []
for ym in sorted(results):
    path = save_parquet(ym, results[ym])
    saved_paths.append(path)

total_rows   = sum(len(df) for df in results.values())

for fname in sorted(os.listdir(OUTPUT_DIR)):
    if fname.endswith(".parquet"):
        fpath = os.path.join(OUTPUT_DIR, fname)
        size_mb = os.path.getsize(fpath) / 1_000_000

/home/spark-af59c1a6-152e-4000-921d-69/.ipykernel/334/command-5086739935280275-911043909:69: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  findf = pd.concat(order_dfs, ignore_index= True)
/home/spark-af59c1a6-152e-4000-921d-69/.ipykernel/334/command-5086739935280275-911043909:69: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  findf = pd.concat(order_dfs, ignore_index= True)
/home/spark-af59c1a6-152e-4000-921d-69/.ipykernel/334/command-5086739935280275-911043909:69: FutureWarning: The behavior of 

In [0]:
schemaCheck = spark.read.parquet("/Volumes/chicago_city/raw/trips_raw/trips_2023-01.parquet")
display(schemaCheck.printSchema())

root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: string (nullable = true)
 |-- trip_end_timestamp: string (nullable = true)
 |-- trip_seconds: double (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_census_tract: double (nullable = true)
 |-- dropoff_census_tract: double (nullable = true)
 |-- pickup_community_area: double (nullable = true)
 |-- dropoff_community_area: double (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- company: string (nullable = true)
 |-- pickup_centroid_latitude: double (nullable = true)
 |-- pickup_centroid_longitude: double (nullable = true)
 |-- pickup_centroid_location: string (nullable = true)
 |-- dropoff_centroid_latitude: double (nullable = true)
 |-- dropoff_centroid